# Getting started with Rerank endpoint

## Overview

Traditional keyword-based search often fails to capture semantic meaning and context, leading to irrelevant results. Cohere's rerank endpoint addresses this by reordering documents based on semantic relevance to a query, improving search accuracy across different languages and reasoning types.

This tutorial demonstrates how to use Cohere's rerank endpoint to improve document ranking through semantic similarity.

By the end, you will be able to implement semantic reranking for various use cases including multilingual queries, numeric comparisons, temporal filtering, and negation handling.

## What we'll cover
- Using the rerank endpoint
- Exploring response object structure
- Implementing multilingual reranking
- Handling numeric reasoning
- Processing temporal reasoning
- Managing negation queries

## Using the Rerank endpoint

In [1]:
import cohere
import os
import yaml
import json
from dotenv import load_dotenv
load_dotenv(override=True)

top_n = 3
#MODEL = "rerank-v4.0-fast"
#co = cohere.ClientV2(os.getenv("COHERE_API_KEY"))

MODEL = os.environ["RERANK_MODEL"]
co = cohere.ClientV2(
    base_url=os.environ["RERANK_BASE_URL"],
    api_key=os.environ["RERANK_API_KEY"],
    timeout=300,
)
query = "Are there fitness-related perks?"

company_faqs = [
    "Reimbursing Travel Expenses: Easily manage your travel expenses by submitting them through our finance tool. Approvals are prompt and straightforward.",
    "Working from Abroad: Working remotely from another country is possible. Simply coordinate with your manager and ensure your availability during core hours.",
    "Health and Wellness Benefits: We care about your well-being and offer gym memberships, on-site yoga classes, and comprehensive health insurance.",
    "Performance Reviews Frequency: We conduct informal check-ins every quarter and formal performance reviews twice a year.",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=company_faqs,
    top_n=top_n
)

print(f"Query: {query}\n")
for result in results.results:
    print(f"  Rank {result.index + 1} (relevance score: {result.relevance_score:.4f})")
    print(f"  {company_faqs[result.index]}\n")

Query: Are there fitness-related perks?

  Rank 3 (relevance score: 0.9435)
  Health and Wellness Benefits: We care about your well-being and offer gym memberships, on-site yoga classes, and comprehensive health insurance.

  Rank 2 (relevance score: 0.5250)
  Working from Abroad: Working remotely from another country is possible. Simply coordinate with your manager and ensure your availability during core hours.

  Rank 4 (relevance score: 0.5133)
  Performance Reviews Frequency: We conduct informal check-ins every quarter and formal performance reviews twice a year.



## Multilingual reranking

In [2]:
def return_results(results, documents):
    for idx, result in enumerate(results.results):
        print(f"Rank: {idx+1}")
        print(f"Score: {result.relevance_score}")
        print(f"Document: {documents[result.index]}\n")
        
# Define the query
query = "피트니스 관련 혜택이 있나요?"  # Are there fitness benefits?

# Rerank the documents
results = co.rerank(
    model=MODEL,
    query=query,
    documents=company_faqs,
    top_n=top_n,
)

return_results(results, company_faqs)

Rank: 1
Score: 0.8907132
Document: Health and Wellness Benefits: We care about your well-being and offer gym memberships, on-site yoga classes, and comprehensive health insurance.

Rank: 2
Score: 0.62758607
Document: Reimbursing Travel Expenses: Easily manage your travel expenses by submitting them through our finance tool. Approvals are prompt and straightforward.

Rank: 3
Score: 0.53276545
Document: Performance Reviews Frequency: We conduct informal check-ins every quarter and formal performance reviews twice a year.



## Numeric reasoning

In [3]:
query = "standing desk shorter than 130 cm"

documents = [
    "Standing desk 125 cm, adjustable height",
    "Standing desk 100 cm, compact design",
    "Standing desk 135 cm, with cable tray",
    "Standing desk 140 cm, dual motor",
    "Standing desk 120 cm, white frame"
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

return_results(results, documents)


Rank: 1
Score: 0.9519705
Document: Standing desk 120 cm, white frame

Rank: 2
Score: 0.9484609
Document: Standing desk 100 cm, compact design

Rank: 3
Score: 0.9463195
Document: Standing desk 125 cm, adjustable height

Rank: 4
Score: 0.68729424
Document: Standing desk 135 cm, with cable tray

Rank: 5
Score: 0.62758607
Document: Standing desk 140 cm, dual motor



## Temporal reasoning

In [4]:
query = "list all my flights between May 1st and June 15th"

documents = [
    "Itinerary: Flight AA123, New York (JFK) to London (LHR), Departure: April 28, 2024, 7:00 PM, Seat 14A",
    "E-Ticket: Flight LH456, Paris (CDG) to Berlin (TXL), Departure: May 10, 2024, 9:30 AM, Seat 22C",
    "Booking Confirmation: Flight JL789, Tokyo (HND) to Seoul (ICN), Departure: May 25, 2024, 2:15 PM, Seat 18B",
    "Itinerary: Flight UA321, Los Angeles (LAX) to San Francisco (SFO), Departure: June 20, 2024, 11:00 AM, Seat 10D",
    "E-Ticket: Flight AZ654, Rome (FCO) to Madrid (MAD), Departure: June 5, 2024, 6:45 AM, Seat 7F",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

return_results(results, documents)


Rank: 1
Score: 0.9827969
Document: Booking Confirmation: Flight JL789, Tokyo (HND) to Seoul (ICN), Departure: May 25, 2024, 2:15 PM, Seat 18B

Rank: 2
Score: 0.9797918
Document: E-Ticket: Flight LH456, Paris (CDG) to Berlin (TXL), Departure: May 10, 2024, 9:30 AM, Seat 22C

Rank: 3
Score: 0.9514318
Document: E-Ticket: Flight AZ654, Rome (FCO) to Madrid (MAD), Departure: June 5, 2024, 6:45 AM, Seat 7F

Rank: 4
Score: 0.7383673
Document: Itinerary: Flight AA123, New York (JFK) to London (LHR), Departure: April 28, 2024, 7:00 PM, Seat 14A

Rank: 5
Score: 0.71191186
Document: Itinerary: Flight UA321, Los Angeles (LAX) to San Francisco (SFO), Departure: June 20, 2024, 11:00 AM, Seat 10D



In [5]:
# flights that occur before the specified month and not the month itself. 
query = "What flights do I have before May?"


results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

return_results(results, documents)

Rank: 1
Score: 0.9825308
Document: Itinerary: Flight AA123, New York (JFK) to London (LHR), Departure: April 28, 2024, 7:00 PM, Seat 14A

Rank: 2
Score: 0.75750744
Document: Booking Confirmation: Flight JL789, Tokyo (HND) to Seoul (ICN), Departure: May 25, 2024, 2:15 PM, Seat 18B

Rank: 3
Score: 0.7531763
Document: E-Ticket: Flight LH456, Paris (CDG) to Berlin (TXL), Departure: May 10, 2024, 9:30 AM, Seat 22C

Rank: 4
Score: 0.6312306
Document: Itinerary: Flight UA321, Los Angeles (LAX) to San Francisco (SFO), Departure: June 20, 2024, 11:00 AM, Seat 10D

Rank: 5
Score: 0.60729057
Document: E-Ticket: Flight AZ654, Rome (FCO) to Madrid (MAD), Departure: June 5, 2024, 6:45 AM, Seat 7F



## Negation

In [6]:
query = "salad without walnuts"

documents = [
    "Spinach salad with walnuts and cranberries",
    "Arugula salad with goat cheese and walnuts",
    "Greek salad with feta and olives",
    "Caesar salad with parmesan and croutons",
    "Tomato basil soup with garlic bread",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

return_results(results, documents)


Rank: 1
Score: 0.7966724
Document: Caesar salad with parmesan and croutons

Rank: 2
Score: 0.78239655
Document: Greek salad with feta and olives

Rank: 3
Score: 0.44707397
Document: Arugula salad with goat cheese and walnuts

Rank: 4
Score: 0.44321477
Document: Tomato basil soup with garlic bread

Rank: 5
Score: 0.4355173
Document: Spinach salad with walnuts and cranberries



## Conclusion

This tutorial covered how to implement Cohere's rerank endpoint for semantic document ranking, including multilingual support, numeric and temporal reasoning, and negation handling. The rerank endpoint provides a powerful way to improve search relevance by moving beyond simple keyword matching to semantic understanding.